In [1]:
from ingest import load_data
documents = load_data()

In [2]:
documents[0]

{'id': '2',
 'recipe_name': "Sarah's Homemade Applesauce",
 'total_time': '25 mins',
 'servings': '4',
 'ingredients': '4  apples - peeled, cored and chopped, ¾ cup water, ¼ cup white sugar, ½ teaspoon ground cinnamon',
 'directions': "Combine apples, water, sugar, and cinnamon in a saucepan; cover and cook over medium heat until apples are soft, about 15 to 20 minutes.\nAllow apple mixture to cool, then mash with a fork or potato masher until it is the consistency you like.\n\n\n\n\n\n\n\n\n\n\n\nPhoto by cookin'mama.\ncookin mama\n",
 'nutrition': 'Total Fat 0g 0%, Sodium 3mg 0%, Total Carbohydrate 32g 12%, Dietary Fiber 4g 13%, Total Sugars 27g, Protein 0g, Vitamin C 6mg 32%, Calcium 13mg 1%, Iron 0mg 1%, Potassium 150mg 3%'}

In [3]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [81]:
data_gen_instructions = """
You emulate a user who is searching for quick recipes ideas. Questions should contain
information about their available time, available groceries, nutritional goals or dietary preferences 
which correspond to the observed recipe record. 
If user is searching by available groceries they don't need to match the ingredients 
in the recipe completely. 
Dietary and nutritional goals should be reflected as well, not as exact numbers, but rather 
as high or low daily usage values by user preference.
Formulate 2 questions this user might ask to get particular recipe suggested. 

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [82]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [91]:
doc = documents[300]

In [92]:
import json
user_prompt = json.dumps(doc)

In [93]:
doc

{'id': '550',
 'recipe_name': 'Persimmon Oatmeal Cookies',
 'total_time': '25 mins',
 'servings': '24',
 'ingredients': '1 ¾ cups rolled oats, 1 ½ cups sifted all-purpose flour, ½ cup English toffee-flavored baking bits (such as Heath®), 1 teaspoon baking soda, 1 teaspoon salt, ¼ teaspoon ground nutmeg, ½ teaspoon ground cinnamon, ¼ teaspoon ground cloves, 1 cup brown sugar, ¾ cup butter, 1  egg, 1 cup persimmon puree, 1 teaspoon vanilla',
 'directions': 'Preheat oven to 350 degrees F (175 degrees C). Line baking sheets with parchment paper.\nWhisk oats, flour, toffee bits, baking soda, salt, nutmeg, cinnamon, and cloves together in a bowl.\nBeat brown sugar and butter together in a bowl with an electric mixer until creamy, 2 minutes. Beat in egg. Beat persimmon puree and vanilla into butter mixture.\nMix flour mixture into persimmon mixture until dough is just moistened and no streaks of flour or oats remain. Drop spoonfuls of dough 1 1/2 inches apart onto ungreased baking sheets.\nBa

In [94]:
user_prompt

'{"id": "550", "recipe_name": "Persimmon Oatmeal Cookies", "total_time": "25 mins", "servings": "24", "ingredients": "1 \\u00be cups rolled oats, 1 \\u00bd cups sifted all-purpose flour, \\u00bd cup English toffee-flavored baking bits (such as Heath\\u00ae), 1 teaspoon baking soda, 1 teaspoon salt, \\u00bc teaspoon ground nutmeg, \\u00bd teaspoon ground cinnamon, \\u00bc teaspoon ground cloves, 1 cup brown sugar, \\u00be cup butter, 1  egg, 1 cup persimmon puree, 1 teaspoon vanilla", "directions": "Preheat oven to 350 degrees F (175 degrees C). Line baking sheets with parchment paper.\\nWhisk oats, flour, toffee bits, baking soda, salt, nutmeg, cinnamon, and cloves together in a bowl.\\nBeat brown sugar and butter together in a bowl with an electric mixer until creamy, 2 minutes. Beat in egg. Beat persimmon puree and vanilla into butter mixture.\\nMix flour mixture into persimmon mixture until dough is just moistened and no streaks of flour or oats remain. Drop spoonfuls of dough 1 1/2

In [95]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [98]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [99]:
response.output_parsed.questions

['Got about 25 minutes and some oats, flour, eggs, and maybe persimmon puree or even just a sweet fruit on hand — what’s a good easy cookie recipe I can make?',
 'I want a quick dessert that isn’t super high in fat or calories, but still tastes cozy and spiced — any persimmon oatmeal cookie recipe suggestions?']

In [100]:
from evaluation_utils import llm_structured

In [101]:
from evaluation_utils import llm_structured_retry

In [102]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [103]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [104]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/598 [00:00<?, ?it/s]

In [105]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

1196

In [106]:
ground_truth[10]

{'question': 'Got about 20 minutes and a few apples at home — what’s an easy warm dessert I can make that isn’t too heavy?',
 'document': '14'}

In [107]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.43399950000000004

In [108]:
import pandas as pd

In [109]:
df_ground_truth = pd.DataFrame(ground_truth)

In [110]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)